# GoodJob 공고 파싱 sLLM 파인튜닝 (Qwen2.5-7B + Unsloth QLoRA)

채용공고 원문 → 구조화 JSON(JobRequirements) 을 출력하도록 Qwen2.5-7B-Instruct 를 LoRA 로 학습합니다.
정답 라벨은 gpt-4o Structured Output 결과(teacher)입니다.

**실행 전 준비**
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 로컬 PC 에서 만든 `finetune/goodjob_colab.zip` 을 준비 (학습 코드 + 데이터셋)

**순서**: GPU 확인 → 설치 → zip 업로드 → (Drive 연결) → 학습·평가 → 결과 저장

| 데이터 | 예상 시간 (T4) |
|---|---|
| train 208건, 2 epoch, eval 23건 + GGUF 내보내기 | 1.5~2.5시간 (설치 포함) |

Colab 무료 세션은 오래 비워 두면 끊기므로, 결과는 Google Drive 에 저장하는 것을 권장합니다.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Unsloth 설치 (3~5분)
설치 후 경고가 나와도 다음 셀로 진행하면 됩니다. 오류가 나면 **런타임 → 세션 다시 시작** 후 이 셀부터 다시 실행하세요.

In [ ]:
%%capture
!pip install unsloth

## 3. 학습 번들 업로드 (`goodjob_colab.zip`)

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # goodjob_colab.zip 선택
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(".")
!wc -l finetune/dataset/parser_train.jsonl finetune/dataset/parser_eval.jsonl

## 4. Google Drive 연결 (권장)
학습 결과(LoRA, GGUF ~4.7GB, 평가 리포트)를 Drive 에 저장합니다. 브라우저 직접 다운로드보다 안정적입니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/goodjob_finetune"
os.makedirs(SAVE_DIR, exist_ok=True)
print("저장 위치:", SAVE_DIR)

## 5. 학습 설정 (수정하지 않아도 됩니다)
아래 값이 본 학습 설정입니다. **`EXPORT = True` 여야 로컬 PC 에서 쓸 GGUF 모델 파일이 만들어집니다.**

In [ ]:
EPOCHS = 2
EVAL_LIMIT = None    # None = eval 전체(23건)
EXPORT = True        # GGUF q4_k_m 내보내기 (False 면 로컬에서 쓸 모델 파일이 생기지 않음)
SKIP_BASE_EVAL = False

import json
n_eval = sum(1 for _ in open("finetune/dataset/parser_eval.jsonl", encoding="utf-8"))
print(f"EPOCHS={EPOCHS}, 평가 {EVAL_LIMIT or n_eval}건, GGUF 내보내기={'O' if EXPORT else 'X'}")
assert EXPORT, "EXPORT=False 이면 로컬에서 쓸 GGUF 파일이 생기지 않습니다."

## 6. 학습 + 평가 실행
1) 베이스 모델 평가 → 2) LoRA 학습 → 3) 파인튜닝 모델 평가 → 4) GGUF 내보내기 순서로 진행됩니다.

In [ ]:
from finetune.train import run_all

report = run_all(
    "finetune/dataset/parser_train.jsonl",
    "finetune/dataset/parser_eval.jsonl",
    epochs=EPOCHS,
    eval_limit=EVAL_LIMIT,
    skip_base_eval=SKIP_BASE_EVAL,
    export=EXPORT,
)

## 7. 결과 확인

In [ ]:
import json
from finetune.metrics import format_table

print(format_table(report["results"]))
print("\ntrain_loss:", report["train_loss"])
print("\n예측 예시:\n", report["sample_predictions"][0][:1500])

## 8. 결과 저장
Drive 를 연결했다면 Drive 로 복사합니다. 연결하지 않았다면 리포트만 브라우저로 다운로드합니다
(GGUF 는 용량이 커서 Drive 사용을 권장).

In [ ]:
import shutil, glob

if "SAVE_DIR" in globals():
    shutil.copy("finetune_report.json", SAVE_DIR)
    shutil.copytree("goodjob-parser-lora", f"{SAVE_DIR}/goodjob-parser-lora", dirs_exist_ok=True)
    gguf = [g for g in report.get("gguf_files", []) if "Q4_K_M" in g.upper()] or report.get("gguf_files", [])
    if gguf:
        shutil.copy(gguf[0], f"{SAVE_DIR}/goodjob-parser.Q4_K_M.gguf")
        print("GGUF 복사:", gguf[0])
    else:
        print("⚠️ GGUF 파일이 없습니다:", report.get("gguf_error", "EXPORT=False"))
    print("Drive 저장 완료:", os.listdir(SAVE_DIR))
else:
    files.download("finetune_report.json")

## 9. 로컬 PC 에서 사용하기
1. Drive 의 `goodjob-parser.Q4_K_M.gguf` 를 프로젝트의 `finetune/ollama/` 폴더로 다운로드
2. Ollama 설치 후: `ollama create goodjob-parser -f finetune/ollama/Modelfile`
3. 로컬 평가: `python -m finetune.eval_local --models qwen2.5:7b goodjob-parser`
4. `.env` 에 `LOCAL_MODEL_NAME=goodjob-parser`, `PARSE_WITH_LOCAL_LLM=true` 설정
5. `finetune_report.json` 은 `finetune/results/` 에 보관 (포트폴리오 근거 자료)